In [ ]:
!pip install transformers pillow tqdm accelerate qwen-vl-utils sentencepiece protobuf

### Environment Setup & Imports

In [1]:
import os
# MUST be set before importing transformers
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

import json
import torch
import gc
from PIL import Image
from tqdm.notebook import tqdm  # Uses the Jupyter-friendly progress bar

# Verify CUDA is available
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU: NVIDIA GeForce RTX 3090


## Configuration & Dataset Aggregation

In [2]:
import os
import json
from PIL import Image

# --- CONFIGURATION ---
INPUT_FILES = [
    "test_1_wo_spec.json", 
    "test_2_wo_spec.json", 
    "train_wo_spec.json", 
    "validation_wo_spec.json"
]
CACHE_FILE = "grounding_cache.json"

# --- STORAGE & RESUME LOGIC ---
def load_data(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    print(f"Warning: {filepath} not found. Skipping.")
    return []

def load_cache(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_cache(cache, filepath):
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(cache, f, indent=4)

# Aggregate unique images and convert non-JPGs
all_unique_images = set()

for filepath in INPUT_FILES:
    dataset = load_data(filepath)
    dataset_modified = False
    
    for item in dataset:
        if "local_image_path" in item:
            # Ensure path uses standard forward slashes for Linux/RunPod
            orig_path = item["local_image_path"].replace("\\\\", "/").replace("\\", "/")
            clean_path = orig_path
            
            # Check if it is NOT a JPEG
            if not orig_path.lower().endswith(('.jpg', '.jpeg')):
                new_path = os.path.splitext(orig_path)[0] + ".jpg"
                
                # Convert the physical file if it hasn't been converted yet
                if not os.path.exists(new_path) and os.path.exists(orig_path):
                    try:
                        with Image.open(orig_path) as img:
                            # Convert to RGB to drop alpha channels (PNG) or palettes (GIF)
                            rgb_im = img.convert('RGB')
                            rgb_im.save(new_path, 'JPEG', quality=95)
                            print(f"Converted image: {orig_path} -> {new_path}")
                    except Exception as e:
                        print(f"Failed to convert {orig_path}: {e}")
                
                # Update the JSON item so downstream tasks know to look for the .jpg
                item["local_image_path"] = new_path
                clean_path = new_path
                dataset_modified = True
                
            all_unique_images.add(clean_path)
            
    # If we modified any paths, save the updated dataset back to disk
    if dataset_modified:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(dataset, f, indent=4)
        print(f"Updated {filepath} with new .jpg paths.")

unique_images = list(all_unique_images)
print(f"Total unique images found across dataset: {len(unique_images)}")

# Initialize cache
cache = load_cache(CACHE_FILE)
for img in unique_images:
    if img not in cache:
        cache[img] = {"deplot_table": None, "vega_lite_spec": None}
save_cache(cache, CACHE_FILE)
print(f"Total Cached images: {len(cache)}")

Total unique images found across dataset: 1680
Total Cached images: 1680


## DePlot (Numeric Grounding)

In [7]:
pending_deplot = [img for img in unique_images if cache[img].get("deplot_table") is None]

if not pending_deplot:
    print("DePlot pass already complete for all images.")
else:
    print(f"Loading DePlot... {len(pending_deplot)} images pending.")
    from transformers import Pix2StructProcessor, Pix2StructForConditionalGeneration
    
    processor = Pix2StructProcessor.from_pretrained("google/deplot")
    model = Pix2StructForConditionalGeneration.from_pretrained("google/deplot").to("cuda")
    
    # --- BATCH CONFIGURATION ---
    BATCH_SIZE = 8  # Increase to 16 or 32 if VRAM permits
    
    # Process in chunks
    for i in tqdm(range(0, len(pending_deplot), BATCH_SIZE), desc="Running Batched DePlot"):
        batch_paths = pending_deplot[i : i + BATCH_SIZE]
        
        try:
            # Load and convert all images in the batch
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            texts = ["Generate underlying data table of the figure below:"] * len(images)
            
            # Pass the entire batch to the processor and model
            inputs = processor(images=images, text=texts, padding=True, return_tensors="pt").to("cuda")
            predictions = model.generate(**inputs, max_new_tokens=512)
            
            # Decode the entire batch
            tables = processor.batch_decode(predictions, skip_special_tokens=True)
            
            # Save results to cache
            for img_path, table in zip(batch_paths, tables):
                cache[img_path]["deplot_table"] = table
                
            save_cache(cache, CACHE_FILE)
            
        except Exception as e:
            print(f"\nError processing batch starting at {batch_paths[0]}: {e}")
            
    # Critical: Flush VRAM
    del model
    del processor
    gc.collect()
    torch.cuda.empty_cache()
    print("DePlot batched extraction complete! VRAM flushed.")

Loading DePlot... 1426 images pending.


Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

Running DePlot:   0%|          | 0/1426 [00:00<?, ?it/s]

DePlot extraction complete! VRAM flushed.


# To Flush all visual groundings

In [5]:
# import json
# import os

# CACHE_FILE = "grounding_cache.json"

# if os.path.exists(CACHE_FILE):
#     with open(CACHE_FILE, 'r', encoding='utf-8') as f:
#         cache = json.load(f)
        
#     flush_count = 0
#     for img_path in cache:
#         if cache[img_path].get("vega_lite_spec") is not None:
#             cache[img_path]["vega_lite_spec"] = None
#             flush_count += 1
            
#     with open(CACHE_FILE, 'w', encoding='utf-8') as f:
#         json.dump(cache, f, indent=4)
        
#     print(f"Successfully flushed {flush_count} Vega-Lite specs from the cache.")
#     print("DePlot tables were preserved. Ready for fresh VLM inference.")
# else:
#     print(f"{CACHE_FILE} not found.")

Flushed 132 broken/cut-off Vega-Lite specs.
You can now re-run the Qwen cell to fix just these missing charts.


# Removing/fixing broken jsons

In [13]:
!pip install json-repair


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [15]:
import json
import re
from json_repair import repair_json

CACHE_FILE = "grounding_cache.json"

with open(CACHE_FILE, 'r', encoding='utf-8') as f:
    cache = json.load(f)

repaired_count = 0
flush_count = 0

for img_path, data in cache.items():
    spec = data.get("vega_lite_spec")
    if spec:
        # 1. Clean markdown formatting
        clean_spec = re.sub(r'```json\n|\n```|```', '', spec).strip()
        
        try:
            # 2. Try standard load first
            json.loads(clean_spec)
        except json.JSONDecodeError:
            # 3. If standard fails, try to repair it (handles comments, trailing commas, etc.)
            repaired_string = repair_json(clean_spec)
            
            try:
                # Verify the repaired version is now valid
                json.loads(repaired_string)
                cache[img_path]["vega_lite_spec"] = repaired_string
                repaired_count += 1
            except:
                # 4. If it's still broken (e.g. severely cut off), then flush it
                # cache[img_path]["vega_lite_spec"] = None
                flush_count += 1

with open(CACHE_FILE, 'w', encoding='utf-8') as f:
    json.dump(cache, f, indent=4)

print(f"Successfully repaired {repaired_count} specs (e.g., fixed comments/commas).")
print(f"Flushed {flush_count} beyond-repair specs for regeneration.")

Successfully repaired 81 specs (e.g., fixed comments/commas).
Flushed 0 beyond-repair specs for regeneration.


# Qwen2.5-VL (Visual Grounding & Color Info)

In [8]:
pending_qwen = [img for img in unique_images if cache[img].get("vega_lite_spec") is None]

if not pending_qwen:
    print("Qwen2.5-VL pass already complete for all images.")
else:
    print(f"Loading Qwen2.5-VL-3B-Instruct... {len(pending_qwen)} images pending.")
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    from qwen_vl_utils import process_vision_info
    
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct", 
        torch_dtype=torch.bfloat16, 
        device_map="auto"
    )
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
    
    # CRITICAL FIX FOR BATCHED GENERATION
    processor.tokenizer.padding_side = "left"
    
    prompt = """Analyze this chart and output a STRICTLY MINIMIZED Vega-Lite specification in JSON format. 

ONLY include information that is clearly visible.

Extract:
1. Chart type under 'mark' (e.g., bar, line, pie).
2. Axis labels or dimensions under 'encoding' (use 'theta' instead of x/y for pie charts).
3. A brief trend description under 'description'.
4. Color mapping: You MUST extract the color for EVERY specific series, category, or pie slice shown in the chart. Under the 'color' encoding block, you must include a 'scale' object containing:
   - 'domain': an array of all the exact category/series names.
   - 'range': an array of their exact corresponding colors.

Rules:
- If uncertain, use "unknown"
- Do NOT guess or infer missing information
- Do NOT include underlying data points
- Output ONLY valid JSON"""

    # --- BATCH CONFIGURATION ---
    BATCH_SIZE = 4  # Start at 4. Increase to 8 if you have a 48GB GPU.

    for i in tqdm(range(0, len(pending_qwen), BATCH_SIZE), desc="Running Batched Qwen"):
        batch_paths = pending_qwen[i : i + BATCH_SIZE]
        batch_messages = []
        
        try:
            # Construct messages for every image in the batch
            for img_path in batch_paths:
                batch_messages.append([
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": img_path},
                            {"type": "text", "text": prompt},
                        ],
                    }
                ])
                
            # Process the batched messages
            texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]
            image_inputs, video_inputs = process_vision_info(batch_messages)
            
            # padding=True ensures the different length sequences align in the tensor
            inputs = processor(
                text=texts,
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to("cuda")
            
            # Generate batched output
            generated_ids = model.generate(**inputs, max_new_tokens=768)
            
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for out_ids, in_ids in zip(generated_ids, inputs.input_ids)
            ]
            
            outputs = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )
            
            # Map outputs back to cache
            for img_path, output_text in zip(batch_paths, outputs):
                cache[img_path]["vega_lite_spec"] = output_text
                
            save_cache(cache, CACHE_FILE)
            
        except Exception as e:
            print(f"\nError processing batch starting at {batch_paths[0]} with Qwen: {e}")

    save_cache(cache, CACHE_FILE)
    
    del model
    del processor
    gc.collect()
    torch.cuda.empty_cache()
    print("Qwen2.5-VL batched extraction complete! VRAM flushed.")

Loading Qwen2.5-VL-3B-Instruct... 1421 images pending.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Running Batched Qwen:   0%|          | 0/356 [00:00<?, ?it/s]

Qwen2.5-VL batched extraction complete! VRAM flushed.
